In [34]:
!pip install -q librosa soundfile numpy

In [35]:
import numpy as np
import librosa

In [36]:
def extract_noise_features(audio_chunk, sr=16000):
    """
    Takes a single chunk of audio (numpy array) and returns 5 noise features.

    Args:
        audio_chunk : numpy array of shape [samples]  (e.g. 3sec * 16000 = 48000 samples)
        sr          : sample rate (always 16000 in our pipeline)

    Returns:
        numpy array of shape [5]
    """

    # Safety check — if chunk is silence or too short, return zeros
    if len(audio_chunk) < 512 or np.max(np.abs(audio_chunk)) < 1e-6:
        return np.zeros(5, dtype=np.float32)

    # Compute STFT — Short Time Fourier Transform
    # This converts audio from time domain → frequency domain
    # All 5 features are computed from this
    stft = np.abs(librosa.stft(audio_chunk, n_fft=512, hop_length=256))

    # Power spectrum (magnitude squared)
    power_spectrum = stft ** 2

    # ─────────────────────────────────────────
    # FEATURE 1 — SNR Proxy
    # ─────────────────────────────────────────
    # SNR = Signal to Noise Ratio
    # We estimate it by comparing loud frames vs quiet frames
    # Native speakers tend to have cleaner, more consistent signal
    rms = librosa.feature.rms(y=audio_chunk, frame_length=512, hop_length=256)[0]
    sorted_rms = np.sort(rms)
    n = len(sorted_rms)
    noise_floor = np.mean(sorted_rms[:max(1, n//10)])     # bottom 10% = noise
    signal_level = np.mean(sorted_rms[max(1, n*9//10):])  # top 10% = signal
    snr_proxy = signal_level / (noise_floor + 1e-8)
    snr_proxy = float(np.log1p(snr_proxy))  # log scale so it doesn't explode

    # ─────────────────────────────────────────
    # FEATURE 2 — Spectral Entropy
    # ─────────────────────────────────────────
    # Entropy = how "spread out" or "random" the energy is across frequencies
    # High entropy → energy is spread across many frequencies (noisy/uncertain speech)
    # Low entropy  → energy concentrated in specific frequencies (clear speech)
    power_norm = power_spectrum / (power_spectrum.sum(axis=0, keepdims=True) + 1e-8)
    entropy_per_frame = -np.sum(power_norm * np.log(power_norm + 1e-8), axis=0)
    spectral_entropy = float(np.mean(entropy_per_frame))

    # ─────────────────────────────────────────
    # FEATURE 3 — Spectral Flatness
    # ─────────────────────────────────────────
    # Flatness = how "noise-like" vs "tone-like" the signal is
    # 1.0 = pure white noise (completely flat spectrum)
    # 0.0 = pure tone like a whistle (energy at one frequency only)
    # Speech is somewhere in between — accents affect this slightly
    flatness = librosa.feature.spectral_flatness(y=audio_chunk, n_fft=512, hop_length=256)[0]
    spectral_flatness = float(np.mean(flatness))

    # ─────────────────────────────────────────
    # FEATURE 4 — Spectral Centroid
    # ─────────────────────────────────────────
    # Centroid = the "center of mass" of the spectrum
    # = which frequency carries most of the energy on average
    # Higher centroid → brighter, higher frequency dominated speech
    # Different accents have slightly different centroid patterns
    centroid = librosa.feature.spectral_centroid(y=audio_chunk, sr=sr, n_fft=512, hop_length=256)[0]
    spectral_centroid = float(np.mean(centroid))

    # ─────────────────────────────────────────
    # FEATURE 5 — Spectral Bandwidth
    # ─────────────────────────────────────────
    # Bandwidth = how "wide" the spectrum is around the centroid
    # Narrow bandwidth → energy concentrated around one frequency (tonal)
    # Wide bandwidth   → energy spread across many frequencies (more noise-like)
    # Accent and articulation clarity affect this
    bandwidth = librosa.feature.spectral_bandwidth(y=audio_chunk, sr=sr, n_fft=512, hop_length=256)[0]
    spectral_bandwidth = float(np.mean(bandwidth))

    # ─────────────────────────────────────────
    # Stack all 5 into one vector and return
    # ─────────────────────────────────────────
    features = np.array([
        snr_proxy,
        spectral_entropy,
        spectral_flatness,
        spectral_centroid,
        spectral_bandwidth
    ], dtype=np.float32)

    return features

In [38]:
def extract_noise_for_file(wav_path, chunk_index, sr=16000):

    audio, _ = librosa.load(wav_path, sr=sr, mono=True)

    all_features = []

    for (start, end) in chunk_index:
        chunk = audio[start:end]
        feats = extract_noise_features(chunk, sr=sr)
        all_features.append(feats)

    return np.stack(all_features, axis=0)

In [40]:
def save_noise_cache(file_id, noise_matrix, cache_dir):
    """
    Saves the noise feature matrix for one file to cache.

    Args:
        file_id      : unique identifier for the audio file (e.g. 'speaker_001')
        noise_matrix : numpy array of shape [T, 5]
        cache_dir    : path to cache/noise/ folder
    """
    os.makedirs(cache_dir, exist_ok=True)
    save_path = os.path.join(cache_dir, f"{file_id}.npy")
    np.save(save_path, noise_matrix)
    print(f"✅ Saved {file_id} → shape {noise_matrix.shape}")

# Test it
save_noise_cache(
    file_id     = 'speaker_001_test',
    noise_matrix = noise_matrix,
    cache_dir   = '/content/drive/MyDrive/MUPlovers/cache/noise/'
)

✅ Saved speaker_001_test → shape (79, 5)


In [41]:
import numpy as np
import os

In [42]:
def assemble_features(file_id, cache_dir_embeddings, cache_dir_prosody, cache_dir_noise, cache_dir_features):

    # STEP 1 — Load all 3 files
    embeddings = np.load(os.path.join(cache_dir_embeddings, f"{file_id}.npy"))  # [T, 768]
    prosody    = np.load(os.path.join(cache_dir_prosody,    f"{file_id}.npy"))  # [T, P]
    noise      = np.load(os.path.join(cache_dir_noise,      f"{file_id}.npy"))  # [T, 5]

    # STEP 2 — Check T matches
    T_emb  = embeddings.shape[0]
    T_pros = prosody.shape[0]
    T_noi  = noise.shape[0]

    if not (T_emb == T_pros == T_noi):
        raise ValueError(
            f"❌ T MISMATCH for {file_id}! "
            f"embeddings={T_emb}, prosody={T_pros}, noise={T_noi}"
        )

    # STEP 3 — Glue together
    X = np.concatenate([embeddings, prosody, noise], axis=1)

    # STEP 4 — Save
    os.makedirs(cache_dir_features, exist_ok=True)
    save_path = os.path.join(cache_dir_features, f"{file_id}.npy")
    np.save(save_path, X)

    print(f"✅ {file_id} assembled → shape {X.shape}")

    return X

In [46]:
import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import os

In [47]:
class NativeDataset(Dataset):
    """
    PyTorch Dataset for Native vs Non-Native classification.

    Loads feature matrices from cache and returns them with labels.
    Handles padding so all sequences in a batch have the same T.
    """

    def __init__(self, manifest_csv, cache_dir_features):
        """
        Args:
            manifest_csv       : path to train.csv or val.csv
                                 must have columns: file_id, label
            cache_dir_features : path to cache/features/
        """
        self.manifest        = pd.read_csv(manifest_csv)
        self.cache_dir       = cache_dir_features

    def __len__(self):
        # How many audio files total
        return len(self.manifest)

    def __getitem__(self, idx):
        """
        Returns one sample:
        - X     : feature matrix [T, D]
        - label : 0 or 1
        """
        row     = self.manifest.iloc[idx]
        file_id = row['file_id']
        label   = int(row['label'])

        # Load feature matrix from cache
        feat_path = os.path.join(self.cache_dir, f"{file_id}.npy")
        X = np.load(feat_path).astype(np.float32)  # [T, D]

        X     = torch.tensor(X)
        label = torch.tensor(label, dtype=torch.long)

        return X, label

In [48]:
def collate_fn(batch):
    """
    This function is called automatically by DataLoader.
    It takes a list of samples and pads them to the same T.

    Without this, if speaker_001 has T=79 and speaker_002 has T=74,
    PyTorch can't stack them into one batch — sizes don't match.

    This function:
    - finds the max T in the batch
    - pads all shorter sequences with zeros to match max T
    - creates a mask: 1 = real chunk, 0 = padding
    """

    Xs, labels = zip(*batch)

    # Find the longest sequence in this batch
    T_max = max(x.shape[0] for x in Xs)
    D     = Xs[0].shape[1]

    padded_Xs = []
    masks     = []

    for X in Xs:
        T = X.shape[0]
        pad_len = T_max - T

        # Pad with zeros at the bottom
        # [T, D] → [T_max, D]
        if pad_len > 0:
            padding = torch.zeros(pad_len, D)
            X_padded = torch.cat([X, padding], dim=0)
        else:
            X_padded = X

        # Mask: 1 for real chunks, 0 for padding
        mask = torch.cat([
            torch.ones(T),
            torch.zeros(pad_len)
        ])

        padded_Xs.append(X_padded)
        masks.append(mask)

    # Stack everything into batch tensors
    X_batch     = torch.stack(padded_Xs)   # [B, T_max, D]
    mask_batch  = torch.stack(masks)       # [B, T_max]
    label_batch = torch.stack(list(labels))  # [B]

    return X_batch, mask_batch, label_batch

In [53]:
import os
import pandas as pd
import numpy as np

BASE = '/content/drive/MyDrive/Hackenza_MUPlovers/'

# Check chunk_index
chunk_index_path = BASE + 'data/chunk_index.csv'
print("chunk_index exists:", os.path.exists(chunk_index_path))

# Check embeddings folder
emb_dir = BASE + 'cache/embeddings/'
emb_files = os.listdir(emb_dir)
print("Total embedding files:", len(emb_files))
print("Sample file names:", emb_files[:5])

# Check prosody folder
pros_dir = BASE + 'cache/features/prosody/'
pros_files = os.listdir(pros_dir)
print("Total prosody files:", len(pros_files))
print("Sample file names:", pros_files[:5])

# Check processed audio
audio_dir = BASE + 'data/processed/'
audio_files = os.listdir(audio_dir)
print("Total audio files:", len(audio_files))

chunk_index exists: True
Total embedding files: 160
Sample file names: ['288.npy', '294.npy', '298.npy', '299.npy', '309.npy']
Total prosody files: 160
Sample file names: ['288.npy', '294.npy', '298.npy', '299.npy', '309.npy']
Total audio files: 161


In [54]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [55]:
import os

BASE = '/content/drive/MyDrive/Hackenza_MUPlovers/'
print("Found:", os.path.exists(BASE))
print("\nContents:", os.listdir(BASE))

Found: True

Contents: ['data', 'cache']


In [56]:
# Check all subfolders
for root, dirs, files in os.walk(BASE):
    level = root.replace(BASE, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 2:  # only go 2 levels deep
        for file in files[:3]:  # show first 3 files only
            print(f"{indent}  {file}")

/
data/
  chunk_index.csv
  chunk_vad.csv
  val_ids.csv
  processed/
    .gitkeep
    420.wav
    522.wav
cache/
  features/
    288.npy
    294.npy
    298.npy
    prosody/
  embeddings/
    288.npy
    294.npy
    298.npy
  noise/
    288.npy
    294.npy
    298.npy


In [57]:
import os
import pandas as pd
import numpy as np

BASE      = '/content/drive/MyDrive/Hackenza_MUPlovers/'
AUDIO_DIR = BASE + 'data/processed/'
CHUNK_CSV = BASE + 'data/chunk_index.csv'
EMB_DIR   = BASE + 'cache/embeddings/'
PROS_DIR  = BASE + 'cache/features/prosody/'
NOISE_DIR = BASE + 'cache/noise/'
FEAT_DIR  = BASE + 'cache/features/'

print("✅ Paths defined!")

✅ Paths defined!


In [58]:
chunk_index = pd.read_csv(CHUNK_CSV)
print("Columns:", chunk_index.columns.tolist())
print(chunk_index.head(5))
print("Unique files:", chunk_index['file_id'].nunique())

Columns: ['file_id', 'chunk_id', 'start_sec', 'end_sec', 'start_sample', 'end_sample', 'chunk_len_samples', 'is_padded', 'file_total_samples']
   file_id  chunk_id  start_sec  end_sec  start_sample  end_sample  \
0      288         0        0.0      3.0             0       48000   
1      288         1        1.5      4.5         24000       72000   
2      288         2        3.0      6.0         48000       96000   
3      288         3        4.5      7.5         72000      120000   
4      288         4        6.0      9.0         96000      144000   

   chunk_len_samples  is_padded  file_total_samples  
0              48000          0              556800  
1              48000          0              556800  
2              48000          0              556800  
3              48000          0              556800  
4              48000          0              556800  
Unique files: 160


In [59]:
# Pick first file
sample_id = str(chunk_index['file_id'].unique()[0])
print("Testing file:", sample_id)

# Load embedding
emb = np.load(EMB_DIR + f"{sample_id}.npy")
print("Embedding shape:", emb.shape)  # should be [T, 768]

Testing file: 288
Embedding shape: (24, 768)


In [60]:
pros = np.load(PROS_DIR + f"{sample_id}.npy")
print("Prosody shape:", pros.shape)  # should be [T, 10]

# Check T matches
print("T match:", emb.shape[0] == pros.shape[0])

Prosody shape: (24, 10)
T match: True


In [61]:
import librosa

# Get chunks for file 288
file_chunks = chunk_index[chunk_index['file_id'] == 288]
chunk_boundaries = list(zip(file_chunks['start_sample'], file_chunks['end_sample']))

# Extract noise
audio, _ = librosa.load(AUDIO_DIR + '288.wav', sr=16000, mono=True)
noise_matrix = np.stack([
    extract_noise_features(audio[s:e]) for (s, e) in chunk_boundaries
], axis=0)

print("Noise matrix shape:", noise_matrix.shape)  # should be (24, 5)
print("T match with embeddings:", noise_matrix.shape[0] == emb.shape[0])

Noise matrix shape: (24, 5)
T match with embeddings: True


In [62]:
from tqdm import tqdm

os.makedirs(NOISE_DIR, exist_ok=True)

all_file_ids = chunk_index['file_id'].unique()
failed = []

for file_id in tqdm(all_file_ids):
    try:
        # Get chunk boundaries for this file
        file_chunks = chunk_index[chunk_index['file_id'] == file_id]
        chunk_boundaries = list(zip(file_chunks['start_sample'], file_chunks['end_sample']))

        # Load audio
        audio, _ = librosa.load(AUDIO_DIR + f"{file_id}.wav", sr=16000, mono=True)

        # Extract noise features
        noise_matrix = np.stack([
            extract_noise_features(audio[s:e]) for (s, e) in chunk_boundaries
        ], axis=0)

        # Save to cache
        np.save(NOISE_DIR + f"{file_id}.npy", noise_matrix)

    except Exception as e:
        print(f"❌ Failed: {file_id} → {e}")
        failed.append(file_id)

print(f"\n✅ Done! {len(all_file_ids) - len(failed)}/{len(all_file_ids)} files processed")
if failed:
    print("❌ Failed files:", failed)

 11%|█         | 17/160 [00:13<01:52,  1.27it/s]


KeyboardInterrupt: 

In [ ]:
from tqdm import tqdm

failed = []

for file_id in tqdm(chunk_index['file_id'].unique()):
    try:
        X = assemble_features(
            file_id              = str(file_id),
            cache_dir_embeddings = EMB_DIR,
            cache_dir_prosody    = PROS_DIR,
            cache_dir_noise      = NOISE_DIR,
            cache_dir_features   = FEAT_DIR
        )
    except Exception as e:
        print(f"❌ Failed: {file_id} → {e}")
        failed.append(file_id)

print(f"\n✅ Done! {len(chunk_index['file_id'].unique()) - len(failed)}/160 files assembled")
if failed:
    print("❌ Failed files:", failed)

In [ ]:
# Check all 160 feature files exist and shapes are correct
all_file_ids = chunk_index['file_id'].unique()
shapes = []
issues = []

for file_id in all_file_ids:
    path = FEAT_DIR + f"{file_id}.npy"
    if not os.path.exists(path):
        issues.append(f"MISSING: {file_id}")
        continue
    X = np.load(path)
    if X.shape[1] != 783:
        issues.append(f"WRONG D: {file_id} → {X.shape}")
        continue
    shapes.append(X.shape[0])

print(f"✅ Total files: {len(shapes)}/160")
print(f"✅ Feature dim D: 783 for all files")
print(f"✅ T range: min={min(shapes)}, max={max(shapes)}, avg={sum(shapes)//len(shapes)}")

if issues:
    print("❌ Issues found:", issues)
else:
    print("✅ No issues! Ready to hand off to H!")

In [63]:
from sklearn.preprocessing import StandardScaler
import numpy as np
import os

def normalize_and_save(feat_dir, output_dir):
    """
    Loads all feature files, fits a StandardScaler on ALL of them,
    then saves normalized versions to output_dir.

    Why: wav2vec features are on a completely different scale than
    noise/prosody features. Without this the GRU ignores small-valued
    features entirely.
    """

    os.makedirs(output_dir, exist_ok=True)

    all_file_ids = [f.replace('.npy', '') for f in os.listdir(feat_dir) if f.endswith('.npy')]

    # ─────────────────────────────────────────
    # STEP 1 — Load all features and stack
    # into one big matrix to fit the scaler
    # ─────────────────────────────────────────
    print("Loading all features to fit scaler...")
    all_chunks = []

    for file_id in all_file_ids:
        X = np.load(os.path.join(feat_dir, f"{file_id}.npy"))  # [T, 783]
        all_chunks.append(X)

    # Stack all chunks from all files → [total_chunks, 783]
    all_chunks_stacked = np.vstack(all_chunks)
    print(f"Total chunks across all files: {all_chunks_stacked.shape}")

    # ─────────────────────────────────────────
    # STEP 2 — Fit scaler on ALL chunks
    # This learns mean and std for each of 783 features
    # ─────────────────────────────────────────
    print("Fitting StandardScaler...")
    scaler = StandardScaler()
    scaler.fit(all_chunks_stacked)
    print("✅ Scaler fitted!")

    # ─────────────────────────────────────────
    # STEP 3 — Transform and save each file
    # ─────────────────────────────────────────
    print("Normalizing and saving...")
    for file_id in all_file_ids:
        X = np.load(os.path.join(feat_dir, f"{file_id}.npy"))  # [T, 783]
        X_normalized = scaler.transform(X)                      # [T, 783] normalized
        np.save(os.path.join(output_dir, f"{file_id}.npy"), X_normalized.astype(np.float32))

    print(f"✅ All {len(all_file_ids)} files normalized and saved!")

    return scaler

In [64]:
BASE = '/content/drive/MyDrive/Hackenza_MUPlovers/'

scaler = normalize_and_save(
    feat_dir   = BASE + 'cache/features/',
    output_dir = BASE + 'cache/features_normalized/'
)

Loading all features to fit scaler...
Total chunks across all files: (5179, 783)
Fitting StandardScaler...
✅ Scaler fitted!
Normalizing and saving...
✅ All 160 files normalized and saved!


In [65]:
import pickle

# Save scaler so H can use same normalization on test set
scaler_path = BASE + 'cache/scaler.pkl'
with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)

print("✅ Scaler saved at:", scaler_path)

✅ Scaler saved at: /content/drive/MyDrive/Hackenza_MUPlovers/cache/scaler.pkl


In [66]:
# Check a normalized file looks correct
sample = np.load(BASE + 'cache/features_normalized/288.npy')
print("Shape:", sample.shape)           # should be [T, 783]
print("Mean:", sample.mean().round(4))  # should be close to 0
print("Std:", sample.std().round(4))    # should be close to 1

Shape: (24, 783)
Mean: 0.0018
Std: 1.2613
